In [1]:
import os, re
import opencc
from tqdm import tqdm

os.makedirs("corpus", exist_ok=True)
cc = opencc.OpenCC('t2s')
DATA_DIR = "corpus/datasets/endfield_data"
RAW_EF_PATH = "corpus/raw_ef.txt"
SENTENCE_PATTERN = re.compile(r'([^。！？；\n]+[。！？；\n]+)')

## Load Data & Generate Corpus

In [2]:
print("Loading Endfield raw text...")
raw_chunks = []
if os.path.exists(DATA_DIR):
    for d, _, files in os.walk(DATA_DIR):
        for file in tqdm(files, desc="Scanning EF Files"):
            if not file.endswith('.txt'): continue
            with open(os.path.join(d, file), 'r', encoding='utf-8') as f:
                raw_chunks.append(f.read())

full_raw = '\n'.join(raw_chunks)
print(f"Raw text concatenation complete: total length {len(full_raw)} characters")

Loading Endfield raw text...


Scanning EF Files: 0it [00:00, ?it/s]
Scanning EF Files: 100%|██████████| 68/68 [00:00<00:00, 1463.65it/s]

Raw text concatenation complete: total length 641771 characters


In [3]:
print("Executing batch traditional-to-simplified conversion (single OpenCC call)...")
full_simp = cc.convert(full_raw)  # Single global call for performance

print("Sentence splitting and length filtering...")
sentences = SENTENCE_PATTERN.findall(full_simp)
valid_lines = []
for sent in sentences:
    clean = re.sub(r'\s+', ' ', sent).strip()
    if 20 <= len(clean) <= 200:
        valid_lines.append(clean)

with open(RAW_EF_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(valid_lines) + '\n')
print(f"Endfield processing complete: {len(valid_lines)} lines -> {RAW_EF_PATH}")

Executing batch traditional-to-simplified conversion (single OpenCC call)...
Sentence splitting and length filtering...
Endfield processing complete: 13096 lines -> corpus/raw_ef.txt
